<a href="https://colab.research.google.com/github/alitourani/vision-paper-hub/blob/main/categories/semseg/yolo26/yolov26_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **👁️‍🗨️ Vision Paper Hub** / **Segmentation, Classification, and Object Detection** / **YOLOv26**

This colaboratory is made to assess the quality of the **YOLOv26** model for Human Action Recognition.

- 🚀 [Benchmark](https://github.com/alitourani/vision-paper-hub/tree/main/categories/keypoint-detector/detectron-human-action)
- 🔗 [GitHub Repo](https://github.com/ultralytics/ultralytics)
- 📄 [Paper](https://doi.org/10.48550/arXiv.2509.25164)

## **I. Install the Library**

- First, you need to clone and install the framework:

- ⚠️ You might see a "Restart Session" warning during the first run in Google Colab due to library version mismatches. This is expected! Accept the restart, re-run this cell, and continue!

In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 6.9 MB/s eta 0:00:00


## **II. Object Detection on Video**

In [8]:
import cv2 as cv
from ultralytics import YOLO

def load_yolo_model(model_name: str) -> YOLO:
    """
    Loads a YOLO model from the given model name.
    """
    print(f"Loading YOLO model: {model_name}")
    return YOLO(model_name)

def process_single_frame(model: YOLO, frame: cv.Mat, conf_threshold: float) -> cv.Mat:
    """
    Performs object detection on a single frame and returns the frame with bounding boxes drawn.
    """
    # Perform inference
    results = model.predict(source=frame, conf=conf_threshold, verbose=False)

    # Plot results on the frame
    annotated_frame = results[0].plot()
    return annotated_frame

In [12]:
def run_video_detection(video_path: str, output_filename: str, model: YOLO, conf_threshold: float):
    """
    Performs object detection on a video from a URL, frame by frame,
    and saves the output to a new video file.
    """
    print(f"Starting video processing for: {video_path}")

    cap = cv.VideoCapture(video_path)
    if not cap.isOpened():
      print(f"Error: Could not open video {video_path}")
      return

    # Get video properties
    fps = int(cap.get(cv.CAP_PROP_FPS))
    width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))

    print(f"Video properties: FPS={fps}, Width={width}, Height={height}, Frames={total_frames}")

    # Define the codec and create VideoWriter object
    fourcc = cv.VideoWriter_fourcc(*'mp4v') # Codec for .mp4 files
    out = cv.VideoWriter(output_filename, fourcc, fps, (width, height))

    if not out.isOpened():
        print(f"Error: Could not create video writer for {output_filename}")
        cap.release()
        return

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        if frame_count % (fps * 5) == 0:
            print(f"Processing frame {frame_count}/{total_frames}")
        # Process the frame
        annotated_frame = process_single_frame(model, frame, conf_threshold)
        # Write the annotated frame to the output video
        out.write(annotated_frame)

    # Release everything when job is finished
    cap.release()
    out.release()
    print(f"Video processing complete. Output saved to: {output_filename}")


In [14]:
import requests
import os

# Variables
video_url = 'https://cdn.pixabay.com/video/2016/02/14/2165-155327596_large.mp4'
output_video_filename = 'yolo26_detect_modular.mp4'
model_name = "yolo26n.pt"
confidence_threshold = 0.3

# --- Download the video from URL to Colab local storage (/content/) ---
video_filename = video_url.split('/')[-1] # Extract filename from URL
local_video_path = f'/content/{video_filename}'

print(f"Downloading video from {video_url} to {local_video_path}")
try:
    response = requests.get(video_url, stream=True)
    response.raise_for_status() # Raise an exception for HTTP errors
    with open(local_video_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Video download complete.")
except requests.exceptions.RequestException as e:
    print(f"Error downloading video: {e}")
    local_video_path = None # Indicate that download failed

if local_video_path:
    # Load the YOLO model once
    yolo_model = load_yolo_model(model_name)

    # Run the video detection using the modular functions, passing the local path
    # The output video will also be saved locally in /content/
    output_local_path = f'/content/{output_video_filename}'
    run_video_detection(local_video_path, output_local_path, yolo_model, confidence_threshold)
    print(f"Processed video saved locally: {output_local_path}")
else:
    print("Video download failed, cannot proceed with detection.")

Video download complete.
Loading YOLO model: yolo26n.pt
Starting video processing for: /content/2165-155327596_large.mp4
Video properties: FPS=50, Width=1280, Height=720, Frames=3000
Processing frame 250/3000
Processing frame 500/3000
Processing frame 750/3000
Processing frame 1000/3000
Processing frame 1250/3000
Processing frame 1500/3000
Processing frame 1750/3000
Processing frame 2000/3000
Processing frame 2250/3000
Processing frame 2500/3000
Processing frame 2750/3000
Processing frame 3000/3000
Video processing complete. Output saved to: /content/yolo26_detect_modular.mp4
Processed video saved locally: /content/yolo26_detect_modular.mp4
